In [18]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import classification_report, accuracy_score

# 1. Load your dataset
df = pd.read_csv('synthetic_health_risk_dataset_1000.csv')

# 2. Select the Features relevant to these 3 diseases
# We use BMI and nutritional intake as the main drivers
X = df[['age', 'bmi', 'sugar_g', 'sodium_mg', 'fat_g', 'carbs_g', 'protein_g']]

# 3. Select the Targets
y = df[['diabetes_risk', 'hyprtension_risk', 'obesity_risk']]

# 4. Split into Train (80%) and Test (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. Scale the data (Crucial for medical accuracy)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Data prepared and scaled successfully.")

Data prepared and scaled successfully.


In [19]:
# Initialize the base XGBoost model
base_model = XGBClassifier(
    n_estimators=200, 
    learning_rate=0.05, 
    max_depth=5, 
    use_label_encoder=False, 
    eval_metric='logloss'
)

# Wrap it in a MultiOutput Classifier
multi_model = MultiOutputClassifier(base_model)

# Train the model
multi_model.fit(X_train_scaled, y_train)

print("Model training for Diabetes, Hypertension, and Obesity complete.")

Model training for Diabetes, Hypertension, and Obesity complete.


C:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\training.py:199: UserWarning: [17:15:29] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\training.py:199: UserWarning: [17:15:29] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\training.py:199: UserWarning: [17:15:30] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [23]:
# Sample testing data representing 5 different patient profiles
sample_patients = [
    {
        "description": "High Risk Patient (Obese/Diabetes Profile)",
        "data": [[55, 34.5, 95, 3800, 110, 350, 60]] 
        # [age, bmi, sugar_g, sodium_mg, fat_g, carbs_g, protein_g]
    },
    {
        "description": "Fit Young Adult (Healthy Profile)",
        "data": [[24, 22.1, 25, 1500, 50, 200, 80]]
    },
    {
        "description": "Elderly with High Salt Intake (Hypertension Profile)",
        "data": [[70, 26.8, 40, 5500, 70, 250, 55]]
    },
    {
        "description": "Average Office Worker (Sedentary/Moderate Risk)",
        "data": [[38, 28.2, 65, 3200, 85, 300, 65]]
    },
    {
        "description": "Athlete (High Calorie/Low Risk)",
        "data": [[29, 23.5, 35, 2200, 90, 450, 140]]
    }
]

# Code to run all 5 through your model
print("--- BATCH TEST RESULTS ---")
for patient in sample_patients:
    # Scale the individual data
    scaled_input = scaler.transform(patient['data'])
    
    # Get prediction from your MultiOutput XGBoost model
    prediction = multi_model.predict(scaled_input)[0]
    
    print(f"\nTarget Profile: {patient['description']}")
    print(f"Prediction (D, H, O): {prediction}")
    # Logic: 1 = High Risk, 0 = Low Risk

--- BATCH TEST RESULTS ---

Target Profile: High Risk Patient (Obese/Diabetes Profile)
Prediction (D, H, O): [1 1 1]

Target Profile: Fit Young Adult (Healthy Profile)
Prediction (D, H, O): [0 0 0]

Target Profile: Elderly with High Salt Intake (Hypertension Profile)
Prediction (D, H, O): [0 1 0]

Target Profile: Average Office Worker (Sedentary/Moderate Risk)
Prediction (D, H, O): [1 1 0]

Target Profile: Athlete (High Calorie/Low Risk)
Prediction (D, H, O): [0 0 0]


C:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWa

saving the brain

In [24]:
import joblib

# Save the trained model
joblib.dump(multi_model, 'nutriscan_risk_model.pkl')

# Save the scaler (ESSENTIAL: the app must scale data the same way)
joblib.dump(scaler, 'nutriscan_scaler.pkl')

print("Model and Scaler saved as .pkl files. These are your 'final' artifacts.")

Model and Scaler saved as .pkl files. These are your 'final' artifacts.


Reccomndation Logic (part 2)

In [25]:
def get_advanced_recommendations(predictions, user_data):
    """
    predictions: list [D, H, O] where 1 is High Risk
    user_data: dictionary or Series with nutrient values
    """
    d_risk, h_risk, o_risk = predictions
    recommendations = []
    
    # --- COMBINATION 1: THE TRIPLE THREAT (D + H + O) ---
    if d_risk and h_risk and o_risk:
        recommendations.append(" CRITICAL: Metabolic Syndrome Risk Detected.")
        recommendations.append("- You must follow a 'DASH-Diabetes' combined diet.")
        recommendations.append("- Total Calorie restriction is mandatory. Avoid all processed Sri Lankan snacks (Short-eats, Papadam).")
        recommendations.append("- Priority: Replace all white grains with Kurakkan or Red Rice immediately.")

    # --- COMBINATION 2: DIABETES + HYPERTENSION ---
    elif d_risk and h_risk:
        recommendations.append(" HIGH RISK: Cardiovascular & Blood Sugar Warning.")
        recommendations.append("- Focus on high-fiber, zero-salt meals. Use lime, ginger, and garlic for flavor instead of salt/sugar.")
        recommendations.append("- Avoid 'Bakery items' (Bread/Buns) as they are high in both hidden sodium and refined carbs.")

    # --- COMBINATION 3: DIABETES + OBESITY ---
    elif d_risk and o_risk:
        recommendations.append(" HIGH RISK: Diabesity Profile.")
        recommendations.append("- Focus on 'Glycemic Load'. Reduce portion sizes of carbohydrates by 50%.")
        recommendations.append("- Increase protein intake to maintain muscle while losing fat.")

    # --- COMBINATION 4: HYPERTENSION + OBESITY ---
    elif h_risk and o_risk:
        recommendations.append("HIGH RISK: Heart Strain Warning.")
        recommendations.append("- Immediate Sodium reduction to <1500mg/day.")
        recommendations.append("- Aerobic exercise (walking) is critical to reduce pressure on the heart.")

    # --- INDIVIDUAL RISKS ---
    elif d_risk:
        recommendations.append("ALERT: Prediabetic Patterns Detected.")
        if user_data['sugar_g'] > 50:
            recommendations.append(f"- Your sugar ({user_data['sugar_g']}g) is double the daily limit. Stop added sugar in tea/coffee.")
        recommendations.append("- Increase fiber via Gotu Kola or green leafy salads.")

    elif h_risk:
        recommendations.append(" ALERT: Hypertension Risk.")
        recommendations.append("- Reduce Sodium. Check 'hidden salts' in dried fish (Karawala) and sauces.")
        recommendations.append("- Increase Potassium intake (Bananas/Coconut water) to balance sodium.")

    elif o_risk:
        recommendations.append(" ALERT: Weight Management Required.")
        recommendations.append(f"- Your BMI of {user_data['bmi']} is in the Obesity range. Aim for a 500-calorie daily deficit.")
    
    # --- HEALTHY CASE ---
    else:
        recommendations.append(" EXCELLENT: No major risks detected.")
        recommendations.append("- Continue your current nutritional balance and active lifestyle.")

    return recommendations

In [28]:
sample_patients = [
    {
        "description": "Patient A: Triple Threat (High D + H + O)",
        "data": [[58, 36.2, 110, 4800, 120, 400, 55]] 
        # Logic: Extreme BMI, Sugar, and Sodium
    },
    {
        "description": "Patient B: The Salty Senior (High Hypertension Risk)",
        "data": [[72, 24.5, 30, 6000, 65, 220, 50]]
        # Logic: Normal BMI but dangerously high salt (HTN only)
    },
    {
        "description": "Patient C: Diabesity Profile (High Diabetes + Obesity)",
        "data": [[45, 33.1, 95, 2100, 90, 380, 70]]
        # Logic: High Sugar and BMI, but salt is okay
    },
    {
        "description": "Patient D: The Athlete (Healthy/High Calorie)",
        "data": [[28, 22.5, 30, 2000, 85, 450, 150]]
        # Logic: High Carbs/Protein but healthy BMI/Sugar
    },
    {
        "description": "Patient E: Borderline Office Worker (Obesity Only)",
        "data": [[35, 30.5, 55, 3100, 80, 290, 65]]
        # Logic: Just entered Obesity range, other factors moderate
    }
]

In [29]:
print("="*70)
print("FINAL VALIDATION: MULTI-RISK PREDICTION & LOGIC ENGINE")
print("="*70)

for patient in sample_patients:
    # 1. Transform data
    raw_vals = patient['data']
    scaled_vals = scaler.transform(raw_vals)
    
    # 2. AI Prediction [Diabetes, HTN, Obesity]
    pred = multi_model.predict(scaled_vals)[0]
    
    # 3. Generate Logic-Based Advice
    # Passing the raw list [age, bmi, sugar, sodium, fat, carbs, protein]
    advice = get_advanced_recommendations(pred, raw_vals[0])
    
    # 4. Print Formatted Report
    print(f"\nTEST CASE: {patient['description']}")
    
    # Visual status icons
    d_stat = "🔴" if pred[0] else "🟢"
    h_stat = "🔴" if pred[1] else "🟢"
    o_stat = "🔴" if pred[2] else "🟢"
    
    print(f"RESULTS -> Diabetes: {d_stat} | HTN: {h_stat} | Obesity: {o_stat}")
    print("PERSONALIZED ADVICE:")
    for line in advice:
        print(f"  * {line}")
    print("-" * 70)

FINAL VALIDATION: MULTI-RISK PREDICTION & LOGIC ENGINE

TEST CASE: Patient A: Triple Threat (High D + H + O)
RESULTS -> Diabetes: 🔴 | HTN: 🔴 | Obesity: 🔴
PERSONALIZED ADVICE:
  *  CRITICAL: Metabolic Syndrome Risk Detected.
  * - You must follow a 'DASH-Diabetes' combined diet.
  * - Total Calorie restriction is mandatory. Avoid all processed Sri Lankan snacks (Short-eats, Papadam).
  * - Priority: Replace all white grains with Kurakkan or Red Rice immediately.
----------------------------------------------------------------------

TEST CASE: Patient B: The Salty Senior (High Hypertension Risk)
RESULTS -> Diabetes: 🟢 | HTN: 🔴 | Obesity: 🟢
PERSONALIZED ADVICE:
  *  ALERT: Hypertension Risk.
  * - Reduce Sodium. Check 'hidden salts' in dried fish (Karawala) and sauces.
  * - Increase Potassium intake (Bananas/Coconut water) to balance sodium.
----------------------------------------------------------------------

TEST CASE: Patient C: Diabesity Profile (High Diabetes + Obesity)
RESULTS -

C:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWa